# @wrap_tool_call——工具调用拦截

在工具执行层面实现与 @wrap_model_call 类似的控制——重试、缓存、参数改写、结果后处理。

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

print("导入完成")

导入完成


## 基本结构

In [2]:
@wrap_tool_call
def my_tool_wrapper(request, handler):
    """@wrap_tool_call 的基本结构"""
    # request: 包含工具名、参数、状态等
    result = handler(request)  # 真正执行工具
    return result

print("基本结构：handler(request) 执行工具")

基本结构：handler(request) 执行工具


## 场景 1：工具调用重试

In [3]:
@wrap_tool_call
def retry_tool_on_error(request, handler):
    """工具调用失败时自动重试"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            result = handler(request)
            if attempt > 0:
                print(f" [重试成功] 第 {attempt+1} 次")
            return result
        except Exception as e:
            if attempt < max_retries - 1:
                import time
                time.sleep((attempt + 1) * 2)
                print(f" [重试] 异常，第 {attempt+1} 次重试...")
            else:
                raise
    return None

call_count = 0

@tool
def fetch_course_data(course_id: str) -> str:
    """获取课程数据"""
    global call_count
    call_count += 1
    if call_count < 3:
        raise Exception(f"网络错误（第 {call_count} 次）")
    return f"课程 {course_id}: Python3 基础教程，免费"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[fetch_course_data], middleware=[retry_tool_on_error], system_prompt="你是课程助手。")

result = agent.invoke({"messages": [HumanMessage(content="帮我查一下 python-001")]})
print(f"\n回复: {result['messages'][-1].content}")

print("工具重试中间件已创建")

 [重试] 异常，第 1 次重试...
 [重试] 异常，第 2 次重试...
 [重试成功] 第 3 次

回复: 以下是课程 **python-001** 的信息：

| 项目 | 内容 |
|------|------|
| **课程 ID** | python-001 |
| **课程名称** | 🐍 Python3 基础教程 |
| **价格** | 🆓 **免费** |

这是一门 Python3 的基础入门课程，而且是免费的！如果你对课程内容有任何疑问，或者想进一步了解，随时可以问我哦！😊
工具重试中间件已创建


## 场景 2：修改工具参数

In [4]:
@wrap_tool_call
def normalize_city_name(request, handler):
    """自动规范化城市名称"""
    tool_call = request.tool_call
    if "city" in tool_call.get("args", {}):
        city = tool_call["args"]["city"]
        new_args = {**tool_call["args"], "city": city.strip().replace("　", "")}
        new_tool_call = {**tool_call, "args": new_args}
        request = request.override(tool_call=new_tool_call)
    return handler(request)

print("参数规范化中间件已定义")

参数规范化中间件已定义


## 场景 3：工具结果缓存

In [5]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

tool_cache = {}

@wrap_tool_call
def cache_tool_results(request, handler):
    """缓存工具执行结果"""
    tool_name = request.tool_call.get("name", "unknown")
    tool_args = str(request.tool_call.get("args", {}))
    cache_key = f"{tool_name}:{tool_args}"
    if cache_key in tool_cache:
        print(f"[缓存命中] {tool_name}")
        return ToolMessage(content=tool_cache[cache_key], tool_call_id=request.tool_call.get("id",""), name=tool_name)
    result = handler(request)
    if hasattr(result, 'content'):
        tool_cache[cache_key] = result.content
        print(f"[缓存写入] {tool_name}，共 {len(tool_cache)} 条")
    return result

print("工具缓存中间件已定义")

工具缓存中间件已定义


## 场景 4：工具调用日志与监控

In [6]:
import time
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def monitor_tool_performance(request, handler):
    """监控工具调用的性能"""
    tool_name = request.tool_call.get("name", "unknown")
    tool_args = request.tool_call.get("args", {})
    start = time.time()
    try:
        result = handler(request)
        elapsed = time.time() - start
        print(f"[监控] {tool_name}({tool_args}) 成功，耗时 {elapsed:.2f}s")
        return result
    except Exception as e:
        elapsed = time.time() - start
        print(f"[监控] {tool_name}({tool_args}) 失败，耗时 {elapsed:.2f}s")
        raise

print("性能监控中间件已定义")

性能监控中间件已定义


## 场景 5：根据结果决定后续流程

In [7]:
from langchain.agents.middleware import wrap_tool_call
from langgraph.types import Command
from langchain.messages import AIMessage

@wrap_tool_call
def check_empty_result(request, handler):
    """工具返回空结果时直接结束"""
    result = handler(request)
    if hasattr(result, 'content') and any(kw in str(result.content) for kw in ["未找到", "无结果", "没有"]):
        return Command(update={"messages": [AIMessage(content="抱歉，没有找到相关信息。请换个关键词试试。")]})
    return result

print("空结果处理中间件已定义")

空结果处理中间件已定义


## @wrap_model_call vs @wrap_tool_call

| 维度 | @wrap_model_call | @wrap_tool_call |
| --- | --- | --- |
| 拦截目标 | 模型调用 | 工具执行 |
| 返回类型 | ModelResponse 或 AIMessage | ToolMessage 或 Command |
| 适用场景 | 重试、降级、缓存、prompt 修改 | 重试、缓存、参数改写、结果处理 |

**术语：**

- **@wrap_tool_call**：包裹工具调用的中间件
- **handler(request)**：执行工具的回调
- **Command**：从中间件返回的控制指令，可修改 Agent 状态